In [1]:
from langchain_core.documents import Document


# sample data 

sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [2]:
# # Text Data

# from langchain_community.document_loaders import TextLoader

# loader = TextLoader("data/Python.txt", encoding = "utf-8")

In [3]:
# document = loader.load()

# document

In [4]:
# # PDF Data

# from langchain_community.document_loaders import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()

# document

# Ingestion Pipeline

### Data -> Documents

In [5]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_15416\220481367.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [6]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # Complete File path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("Total pdfs :", num_docs)
    print("Total pages:", len(all_docs))
    return all_docs

In [7]:
all_pdf_docs = load_all_pdfs()

Total pdfs : 2
Total pages: 32


In [8]:
type(all_pdf_docs[0])
type(all_pdf_docs[1])

langchain_core.documents.base.Document

### Chunks

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size = 500, chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size, 
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [10]:
chunks = split_docs(all_pdf_docs)

len(chunks)

321

### Embeddings

In [11]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading Model ....", self.model_name)
        
        self.model = SentenceTransformer(self.model_name)
        print("Embedding Dimensions:", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar = True)
        print("Embedding Shape:", embeddings.shape)
        return embeddings

In [12]:
embedding_manager = EmbeddingManager()

Loading Model .... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimensions: 384


C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_15416\1450699072.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Dimensions:", self.model.get_sentence_embedding_dimension())


### Vector Store - Vector DB smaller local version

In [13]:
import chromadb
import uuid  # helps in creating index value for our values

In [14]:
class VectorStoreManager:
    def __init__(self, persist_directory = "data/vector_store", collection_name = "pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok = True)

        # creating client
        self.client = chromadb.PersistentClient(path = self.persist_directory)

        # creating collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata={"description":"Vector store collection for pdf embeddings in RAG"}
        )

        print("Initialized our Vector Store with collection = ", self.collection_name)
        print("Documents in collection :", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Document length is not same as of Embeddings")

        # store => ids, embeddings, documents, metadata

        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embeddings_list
            )
        print("Total Documents added in vector store :", len(documents_content))
        print("Docs in collection : ",self.collection.count())

In [15]:
vector_store = VectorStoreManager()

Initialized our Vector Store with collection =  pdf_documents
Documents in collection : 0


In [16]:
# Data -> Documents -> Chunks -> Embeddings -> Store in Vector Store

texts = [doc.page_content for doc in chunks]

Embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, Embeddings)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embedding Shape: (321, 384)
Total Documents added in vector store : 321
Docs in collection :  321


# Retrieval Pipeline

In [17]:
# user -> query -> query embedding -> semantic search in Vector Store -> context + query -> LLM -> o/p

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [20]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [21]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_de417f6b-3107-458d-991b-ced2de74620d',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'creator': 'LaTeX with hyperref',
   'moddate': '2024-03-28T00:54:45+00:00',
   'subject': '',
   'author': '',
   'trapped': '/False',
   'producer': 'pdfTeX-1.40.25',
   'content_length': 288,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'page': 0,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'page_label': '1',
   'total_pages': 21,
   'title': '',
   'doc_index': 88,
   'keywords': '',
   'source': 'data/pdfs\\research2.pdf'},
  'distance': 0.4629150927066803,
  'similarity_score': 0.5370849072933197,
  'rank': 1},
 {'id': 'doc_06170ad0-

# Generation Pipeline

## OpenAI - GPT

In [24]:
# pip install langchain-openai

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key = API_KEY_OPENAI,
    model = "gpt-5.4",
    temperature = 0.1,
    max_tokens = 1024
)

In [29]:
# generate our retrieveal augemented output

def generate_output(query, retriever, llm, top_k = 3):
    results = rag_retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("We found no relevant context for the query")

    prompt = f""" use given context to generate the answer for the given query
                    Context: {context}
                    Query: {query}"""
    response = llm.invoke(prompt)  # expects string as prompt
    return response.content

In [ ]:
answer = generate_output("What is RAG?", rag_retriever, llm)

## Groq

In [35]:
# pip install langchain-groq

In [42]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = API_KEY_GROQ,
    model = "llama-3.3-70b-versatile",
    temperature = 0.1,
    max_tokens = 1024
)

In [53]:
# generate our retrieveal augemented output

def generate_output(query, retriever, llm, top_k = 3):
    results = rag_retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("We found no relevant context for the query")

    prompt = f""" use given context to generate the answer for the given query
                    Context: {context}
                    Query: {query}"""
    response = llm.invoke([prompt.format(context = context, query = query)])  # expects list as prompt
    return response.content

In [46]:
answer = generate_output("What is encoder-decoder", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Shape: (1, 384)
retrieved 3 documents


In [47]:
print(answer)

Based on the given context, the encoder-decoder refers to a type of neural network architecture composed of two main components:

1. **Encoder**: A stack of 6 identical layers, each with two sub-layers:
	* The first sub-layer is a multi-head self-attention mechanism.
	* The second sub-layer is a simple, position-wise fully connected feed-forward network.
2. **Decoder**: A stack of 6 identical layers, each with three sub-layers:
	* The first sub-layer is a multi-head self-attention mechanism.
	* The second sub-layer is a multi-head attention mechanism over the output of the encoder stack.
	* The third sub-layer is a simple, position-wise fully connected feed-forward network.

Both the encoder and decoder use residual connections, layer normalization, and positional encodings. The output of each layer in the encoder and decoder has a dimension of d_model = 512. The decoder also has an additional sub-layer that performs multi-head attention over the output of the encoder stack.


## Anthropic - Claude

In [49]:
# pip install langchain-anthropic

In [52]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    anthropic_api_key = API_KEY_CLAUDE,
    model = "claude-haiku-4-5-20251001",
    temperature = 0.1,
    max_tokens = 1024
)

In [54]:
# generate our retrieveal augemented output

def generate_output(query, retriever, llm, top_k = 3):
    results = rag_retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("We found no relevant context for the query")

    prompt = f""" use given context to generate the answer for the given query
                    Context: {context}
                    Query: {query}"""
    response = llm.invoke([prompt.format(context = context, query = query)])  # expects list as prompt
    return response.content

In [ ]:
answer = generate_output("What is encoder-decoder", rag_retriever, llm)